# Physics-Informed Neural Networks for High-Temperature Turbomachinery & Thermodynamic Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/pinn-thermo-dynamics/blob/main/notebooks/quickstart_demo.ipynb)
![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue.svg)
![PyTorch 2.x](https://img.shields.io/badge/PyTorch-2.x-ee4c2c.svg)
![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)

This interactive tutorial demonstrates how to solve and identify transient aerothermal heat transfer dynamics using **pure PyTorch (2.x)** PINNs, incorporating **Xavier initialization**, **Dual-Phase Adam & L-BFGS Optimization**, and **Inverse System Identification**.

## 1. Problem Formulation & Governing Differential Equations

Transient aerothermal heat conduction across a gas turbine blade wall is governed by the 1D unsteady heat conduction PDE:

$$\frac{\partial T}{\partial t} - \alpha \frac{\partial^2 T}{\partial x^2} - Q(x, t) = 0, \quad x \in [0, 1], \; t \in [0, 1]$$

with boundary conditions:
- Exterior hot gas boundary ($x = 0$): $T(0, t) = T_{hot}(t)$
- Internal cooling channel ($x = 1$): $T(1, t) = T_{cool}(t)$
- Initial thermal state ($t = 0$): $T(x, 0) = T_{init}(x)$

Composite scientific loss:

$$\mathcal{L}_{total} = w_{pde} \mathcal{L}_{pde} + w_{bc} \mathcal{L}_{bc} + w_{ic} \mathcal{L}_{ic} + w_{data} \mathcal{L}_{data}$$

In [ ]:
# Environment setup (supports both Google Colab and local environments)
import os, sys
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/your-username/pinn-thermo-dynamics.git
    %cd pinn-thermo-dynamics
    !pip install -r requirements.txt
    sys.path.append(os.getcwd())
else:
    sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import matplotlib.pyplot as plt
from physics.heat_pde import HeatConductionPDE
from physics.boundary_conditions import BoundaryConditionEvaluator
from models.networks import build_network
from models.pinn_solver import PINNSolver
from utils.metrics import compute_metrics
from utils.visualization import plot_solution_tripanel, plot_loss_history, set_academic_style

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active hardware target: {device}")

## 2. Ground Truth & Analytical Manufactured Benchmark

We instantiate an exact analytical reference field with $\alpha_{true} = 0.05$ on a dense spatiotemporal mesh.

In [ ]:
alpha_true = 0.05
decay_lambda = 0.5
pde_forward = HeatConductionPDE(alpha_true=alpha_true, is_inverse=False, decay_lambda=decay_lambda).to(device)
bc_evaluator = BoundaryConditionEvaluator(pde=pde_forward, device=device, seed=42)

# Evaluation grid (256x256)
x_eval = np.linspace(0.0, 1.0, 256)
t_eval = np.linspace(0.0, 1.0, 256)
X_mesh, T_mesh = np.meshgrid(x_eval, t_eval)
T_exact = pde_forward.exact_temperature(X_mesh, T_mesh)
print(f"Ground truth generated on 256x256 grid (65,536 points).")

## 3. Forward Simulation: Hybrid Two-Phase (Adam + L-BFGS) PINN

We train the PINN using **Adam** for global exploration followed by **L-BFGS** for second-order precision.

In [ ]:
model_forward = build_network(
    arch_type="modified_mlp",
    in_dim=2,
    out_dim=1,
    hidden_dim=64,
    num_layers=4,
    activation="tanh",
).to(device)

solver_forward = PINNSolver(
    model=model_forward,
    pde=pde_forward,
    bc_evaluator=bc_evaluator,
    learning_rate_adam=1e-3,
    learning_rate_lbfgs=1.0,
    device=device,
)

print("Phase 1: Running Adam Global Optimization (1000 epochs)...")
for epoch in range(1001):
    m = solver_forward.train_step_adam(epoch=epoch, n_pde=2000, n_ic=128, n_bc=128)
    if epoch % 200 == 0:
        l2 = solver_forward.evaluate_relative_l2(128, 128)
        print(f"Adam Epoch {epoch:4d} | Loss: {m['loss_total']:.2e} | Relative L2: {l2:.3e}")

print("Phase 2: Running L-BFGS Precision Optimization (30 steps)...")
opt_lbfgs = solver_forward._init_lbfgs_optimizer()
for step in range(30):
    m = solver_forward.train_step_lbfgs(opt_lbfgs, step_idx=1001 + step, n_pde=2000, n_ic=128, n_bc=128)
    if step % 10 == 0:
        l2 = solver_forward.evaluate_relative_l2(128, 128)
        print(f"L-BFGS Step {step:2d} | Loss: {m['loss_total']:.2e} | Relative L2: {l2:.3e}")

### Forward Problem Verification: 3-Panel Heatmap

In [ ]:
model_forward.eval()
with torch.no_grad():
    coords = torch.tensor(np.column_stack([X_mesh.ravel(), T_mesh.ravel()]), dtype=torch.float32, device=device)
    T_pred = model_forward(coords).cpu().numpy().reshape(X_mesh.shape)

fig = plot_solution_tripanel(X_mesh, T_mesh, T_exact, T_pred, title_suffix="(PINN Hybrid Adam+L-BFGS)")
plt.show()

metrics_fwd = compute_metrics(T_pred, T_exact)
print(f"Verification Metrics on 256x256 Grid:")
print(f"  * Relative L2 Error : {metrics_fwd['relative_l2_error']:.3e} ({metrics_fwd['relative_l2_error']*100:.2f}%)")
print(f"  * Mean Squared Error: {metrics_fwd['mean_squared_error']:.3e}")
print(f"  * RMSE              : {metrics_fwd['rmse']:.3e}")
print(f"  * Max Error (L_inf) : {metrics_fwd['max_error_linf']:.3e}")

## 4. Inverse Problem: System Identification & Diffusivity Estimation

Here, the thermal diffusivity $\alpha$ is **unknown**. We initialize $\alpha_0 = 0.01$ (a 5x error from the true value $0.05$). The network assimilates sparse noisy sensor observations to discover the true physical parameter.

In [ ]:
pde_inverse = HeatConductionPDE(
    alpha_true=alpha_true,
    is_inverse=True,
    initial_guess_alpha=0.01,
    decay_lambda=decay_lambda,
).to(device)

bc_eval_inv = BoundaryConditionEvaluator(pde=pde_inverse, device=device, seed=42)
model_inverse = build_network(arch_type="modified_mlp", in_dim=2, out_dim=1, hidden_dim=64, num_layers=4).to(device)

solver_inverse = PINNSolver(
    model=model_inverse,
    pde=pde_inverse,
    bc_evaluator=bc_eval_inv,
    learning_rate_adam=1e-3,
    warmup_epochs=300,
    device=device,
)

print("Training Inverse PINN (Estimating unknown thermal diffusivity alpha)...")
alpha_history = []
for epoch in range(1201):
    m = solver_inverse.train_step_adam(epoch=epoch, n_pde=2000, n_ic=128, n_bc=128, n_sensors=150, sensor_noise=0.01)
    alpha_history.append(m["alpha_learned"])
    if epoch % 150 == 0:
        print(f"Epoch {epoch:4d} | Discovered Alpha: {m['alpha_learned']:.5f} (True: {alpha_true}) | Parameter Error: {m['alpha_error_pct']:.2f}%")

alpha_final = pde_inverse.alpha.item()
param_err = abs(alpha_final - alpha_true) / alpha_true * 100.0
print(f"\nFinal Discovered Diffusivity: {alpha_final:.5f}")
print(f"Ground Truth Diffusivity    : {alpha_true:.5f}")
print(f"Parameter Estimation Error  : {param_err:.3f}%")

### Parameter Convergence Plot

In [ ]:
set_academic_style()
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(alpha_history, label=r"Discovered Diffusivity $\hat{\alpha}$", color="#1f77b4", linewidth=2.0)
ax.axhline(alpha_true, label=rf"True Diffusivity $\alpha = {alpha_true}$", color="#d62728", linestyle="--", linewidth=1.8)
ax.set_xlabel("Optimization Epoch")
ax.set_ylabel(r"Diffusivity $\alpha$ [$m^2/s$]")
ax.set_title("Inverse Parameter Calibration: Rapid Discovery of Thermal Diffusivity")
ax.legend(frameon=True)
plt.show()